# Candidate Ranking System - Colab Sandbox
## India Runs Data & AI Challenge 2026

This notebook reproduces the ranking pipeline on a small sample for Stage 3 verification.

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy pandas scikit-learn python-dateutil tqdm python-docx python-pptx

# Clone repo fresh
!rm -rf candidate-ranker-system
!git clone https://github.com/shivv23/candidate-ranker-system.git
%cd candidate-ranker-system

In [ ]:
# Dynamic reload to avoid stale pyc cache
import sys, importlib
if 'src' in sys.modules:
    for m in list(sys.modules.keys()):
        if m.startswith('src'):
            del sys.modules[m]

# Now import fresh
sys.path.insert(0, '.')
from src.loader import load_candidates
from src.feature_extractor import extract_features
from src.embedding_engine import EmbeddingEngine
from src.scorers import (score_title, score_skills, score_experience,
    score_company, score_signals, score_location, score_education,
    detect_honeypot_penalty, compute_final_score)
from src.config import DATA_RAW, SEMANTIC_QUERY
from src.reasoning import generate_reasoning
import pickle, numpy as np, csv

In [ ]:
# Run Phase A: Precompute on sample
candidates = load_candidates(DATA_RAW / 'sample_candidates.json')
features = [extract_features(c) for c in candidates]
print(f'Loaded {len(features)} sample candidates')

texts = [f.profile_text for f in features]
engine = EmbeddingEngine()
embeddings = engine.embed(texts, batch_size=32, show_progress=True)
index = engine.build_index(embeddings)
print(f'Embeddings shape: {embeddings.shape}')

for f in features:
    f.title_score = score_title(f)
    f.skills_score = score_skills(f)
    f.exp_score = score_experience(f)
    f.company_score = score_company(f)
    f.signal_score = score_signals(f)
    f.loc_score = score_location(f)
    f.edu_score = score_education(f)
    f.hon_penalty, f.hon_flags = detect_honeypot_penalty(f)
print('Pre-scoring complete')

In [ ]:
# Phase B: Rank
jd_vec = engine.embed_single(SEMANTIC_QUERY)
scores_raw, indices = engine.search(index, jd_vec, k=len(features))

semantic_by_id = {}
for score, idx in zip(scores_raw, indices):
    semantic_by_id[features[idx].candidate_id] = float(score)

ranked = []
for i in indices:
    f = features[i]
    final = compute_final_score(f, semantic_by_id.get(f.candidate_id, 0.0),
        f.title_score, f.skills_score, f.exp_score, f.company_score,
        f.signal_score, f.loc_score, f.edu_score, f.hon_penalty)
    ranked.append((f.candidate_id, final, f))

ranked.sort(key=lambda x: (-round(x[1], 4), x[0]))
top = ranked[:min(100, len(ranked))]

results = []
for i, (cid, score, f) in enumerate(top):
    scores_dict = {'title': f.title_score, 'skills': f.skills_score,
        'exp': f.exp_score, 'company': f.company_score,
        'signal': f.signal_score, 'location': f.loc_score,
        'edu': f.edu_score, 'semantic': semantic_by_id.get(cid, 0),
        'hon_penalty': f.hon_penalty}
    reasoning = generate_reasoning(f, scores_dict, i + 1)
    results.append({'candidate_id': cid, 'rank': i + 1,
        'score': score, 'reasoning': reasoning})

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['candidate_id', 'rank', 'score', 'reasoning'])
    writer.writeheader()
    writer.writerows(results)

print('Top 5:')
for r in results[:5]:
    print(f"  {r['rank']}. {r['candidate_id']} - {r['score']:.4f}")
print(f'\nSubmission saved to submission.csv')

In [ ]:
# Quick format check (sample has 50 candidates, validator expects 100)
import csv
with open('submission.csv') as f:
    rows = list(csv.DictReader(f))
print('Rows: ' + str(len(rows)))
print('Header: candidate_id, rank, score, reasoning')
r0 = rows[0]
print('First: ' + r0['candidate_id'] + ' - score ' + r0['score'])
rn = rows[-1]
print('Last:  ' + rn['candidate_id'] + ' - score ' + rn['score'])
scores = [float(r['score']) for r in rows]
ok = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))
print('Non-increasing: ' + str(ok))
print()
print('Note: Full validation with 100 rows passes on complete dataset (see rank.py)')